In [ ]:
from pathlib import Path
import os
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import chi2_contingency
import pandas as pd
import statsmodels.api as sm
import numpy as np

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_BATCHES =  PROJECT_ROOT / "data" / "batches"
DATA_OUTPUT =  PROJECT_ROOT / "data" / "output"

In [ ]:
dataset_llama = pd.read_csv(DATA_PROCESSED / "final_dataset_NAs.csv")
dataset_qwen = pd.read_csv(DATA_PROCESSED / "final_dataset_NAs_qwen.csv")

In [ ]:
# ============================================================
# Group rare qualitative LLM categories
# ============================================================

qual_cols = [
    "main_focus",
    "secondary_focus",
    "overall_outlook",
    "managerial_horizon"]

def group_rare_categories(df, cols, min_count=30):
    df = df.copy()

    for col in cols:
        df[col] = df[col].astype(str).str.strip()

        counts = df[col].value_counts()
        keep_categories = counts[counts >= min_count].index

        df[col + "_grouped"] = df[col].where(
            df[col].isin(keep_categories),
            "Other"
        )

    return df


dataset_llama = group_rare_categories(
    dataset_llama,
    qual_cols,
    min_count=30
)

dataset_qwen = group_rare_categories(
    dataset_qwen,
    qual_cols,
    min_count=30
)

datasets = {
    "llama": dataset_llama,
    "qwen": dataset_qwen
}


In [ ]:
# ============================================================
# MULTICOLLINEARITY DIAGNOSTICS
# VIF separately + GVIF separately + Cramer's V separately
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import chi2_contingency
import pandas as pd
import statsmodels.api as sm
import numpy as np


# ============================================================
# HELPERS
# ============================================================

def numerical_vif(df, variables):
    vif_df = df[variables].dropna().copy()

    X = sm.add_constant(vif_df)

    vif_table = pd.DataFrame()
    vif_table["Variable"] = X.columns
    vif_table["VIF"] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]

    return vif_table.sort_values("VIF", ascending=False)


def categorical_vif(df, categorical_vars):
    cat_df = df[categorical_vars].dropna().copy()

    X = pd.get_dummies(
        cat_df,
        columns=categorical_vars,
        drop_first=True,
        dtype=float
    )

    X = sm.add_constant(X)

    vif_table = pd.DataFrame()
    vif_table["Variable"] = X.columns
    vif_table["VIF"] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]

    return vif_table.sort_values("VIF", ascending=False)


def mixed_vif(df, numerical_vars, categorical_vars):
    temp = df[numerical_vars + categorical_vars].dropna().copy()

    dummies = pd.get_dummies(
        temp[categorical_vars],
        drop_first=True,
        dtype=float
    )

    X = pd.concat(
        [temp[numerical_vars], dummies],
        axis=1
    )

    X = sm.add_constant(X)

    vif_table = pd.DataFrame()
    vif_table["Variable"] = X.columns
    vif_table["VIF"] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]

    return vif_table.sort_values("VIF", ascending=False)


def categorical_gvif(df, categorical_vars):
    """
    Approximate GVIF for each grouped categorical variable.
    Report GVIF and adjusted GVIF.
    """

    cat_df = df[categorical_vars].dropna().copy()

    dummies = pd.get_dummies(
        cat_df,
        columns=categorical_vars,
        drop_first=True,
        dtype=float
    )

    gvif_results = []

    for cat in categorical_vars:
        cols = [
            c for c in dummies.columns
            if c.startswith(cat + "_")
        ]

        if len(cols) == 0:
            continue

        corr = dummies[cols].corr()

        try:
            det_corr = np.linalg.det(corr)

            if det_corr <= 0:
                gvif = np.nan
            else:
                gvif = 1 / det_corr

        except Exception:
            gvif = np.nan

        df_cat = len(cols)

        gvif_adj = (
            gvif ** (1 / (2 * df_cat))
            if pd.notnull(gvif)
            else np.nan
        )

        gvif_results.append({
            "Variable": cat,
            "DF": df_cat,
            "GVIF": gvif,
            "GVIF_adj": gvif_adj
        })

    return pd.DataFrame(gvif_results)


def cramers_v_with_p(x, y):
    contingency = pd.crosstab(x, y)

    chi2, p, dof, expected = chi2_contingency(contingency)

    n = contingency.sum().sum()
    phi2 = chi2 / n
    r, k = contingency.shape

    cramers_v = np.sqrt(
        phi2 / min(k - 1, r - 1)
    )

    return cramers_v, p, chi2, dof


def categorical_cramers(df, categorical_vars):
    results = []

    for i, var1 in enumerate(categorical_vars):
        for var2 in categorical_vars[i + 1:]:

            temp = df[[var1, var2]].dropna().copy()

            v, p, chi2, dof = cramers_v_with_p(
                temp[var1],
                temp[var2]
            )

            results.append({
                "Variable 1": var1,
                "Variable 2": var2,
                "Cramers V": v,
                "Chi2": chi2,
                "p-value": p,
                "DOF": dof
            })

    return pd.DataFrame(results)


def save_latex_table(df, caption, label, filename):
    df = df.copy()

    # Remove constant if present
    if "Variable" in df.columns:
        df = df[df["Variable"] != "const"].copy()

    latex = df.to_latex(
        index=False,
        float_format="%.4f",
        escape=False
    )

    latex = (
        "\\begin{table}[htbp]\n"
        "\\centering\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        + latex +
        "\\end{table}"
    )

    with open(multicol_output_dir / filename, "w") as f:
        f.write(latex)


# ============================================================
# VARIABLES
# ============================================================

numerical_vars = [
    "forward_looking_intensity",
    "dict_score",
    "specificity",
    "economic_substance",
    "tone",
    "certainty",
    "bm",
    "log_word_count",
    'log_assets'
]

categorical_vars = [
    "main_focus_grouped",
    "secondary_focus_grouped",
    "managerial_horizon_grouped",
    "overall_outlook_grouped",
    
]


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

multicol_output_dir = DATA_PROCESSED / "multicollinearity_tables"
multicol_output_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# RUN DIAGNOSTICS
# ============================================================

diagnostic_datasets = {
    "llama": dataset_llama,
    "qwen": dataset_qwen
}

for model_name, df in diagnostic_datasets.items():

    # -------------------------
    # VIF: numerical only
    # -------------------------
    vif_num = numerical_vif(
        df,
        numerical_vars
    )

    save_latex_table(
        vif_num,
        caption=f"Numerical VIF diagnostics for the {model_name.capitalize()} specifications",
        label=f"tab:vif_numerical_{model_name}",
        filename=f"vif_numerical_{model_name}.tex"
    )

    # -------------------------
    # VIF: categorical dummies only
    # -------------------------
    vif_cat = categorical_vif(
        df,
        categorical_vars
    )

    save_latex_table(
        vif_cat,
        caption=f"Categorical dummy VIF diagnostics for the {model_name.capitalize()} specifications",
        label=f"tab:vif_categorical_{model_name}",
        filename=f"vif_categorical_{model_name}.tex"
    )

    # -------------------------
    # VIF: numerical + categorical dummies
    # -------------------------
    vif_mix = mixed_vif(
        df,
        numerical_vars,
        categorical_vars
    )

    save_latex_table(
        vif_mix,
        caption=f"Mixed numerical and categorical VIF diagnostics for the {model_name.capitalize()} specifications",
        label=f"tab:vif_mixed_{model_name}",
        filename=f"vif_mixed_{model_name}.tex"
    )

    # -------------------------
    # GVIF: categorical variables only
    # -------------------------
    gvif_cat = categorical_gvif(
        df,
        categorical_vars
    )

    save_latex_table(
        gvif_cat,
        caption=f"GVIF diagnostics for grouped categorical variables in the {model_name.capitalize()} specifications",
        label=f"tab:gvif_categorical_{model_name}",
        filename=f"gvif_categorical_{model_name}.tex"
    )

    # -------------------------
    # Cramer's V
    # -------------------------
    cramers = categorical_cramers(
        df,
        categorical_vars
    )

    save_latex_table(
        cramers,
        caption=f"Cramer's V association statistics for grouped categorical variables in the {model_name.capitalize()} specifications",
        label=f"tab:cramers_{model_name}",
        filename=f"cramers_{model_name}.tex"
    )

print("Saved all multicollinearity diagnostics to:")
print(multicol_output_dir)